#  Problem 2
## Borges and Daripa JCP Paper

### Problem
$$
\begin{align*}
u(x,y) = (x+1)^{5/2}(y+1)^{5/2}-(x+1)(y+1)^{5/2}-(x+1)^{5/2}(y+1)+(x+1)(y+1) \\
f(x,y) = \frac{15}{4}(-((1+x)\sqrt{1+y})+(1+x)^{5/2}\sqrt{1+y}-\sqrt{1+x}(1+y)+\sqrt{1+x}(1+y)^{5/2})
\end{align*}
$$

Unit Disk
table 3, relative errors infimim and l2, fix N=64, vary M, dirichlet and neumann problems trapezoidal and simpsons rule



# Imports

In [1]:
import numpy as np
import warnings
import matplotlib.pyplot as plt
from matplotlib.tri import Triangulation
import numpy as np
import time
import matplotlib.pyplot as plt
import pandas as pd

import os, sys

# Main project root
repo_root = r"C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research"
os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.append(repo_root)


from poisson_solver_testing import (
    generate_grid_values,
    generate_uniform_radial,
    generate_nonuniform_radial,
    generate_uniform_azimuthal,
    generate_fixed_nonuniform_azimuthal,
    generate_nonuniform_azimuthal,
    generate_cartesian_grid_on_disk,
    trap_2d_on_disk,
    compute_error_metrics,
    plot_on_disk_with_error,
    plot_on_disk,
    poisson_solver
)




# Problem Setup

In [ ]:
# Problem Setup  — Problem 2 (Borges–Daripa JCP)

# Radius
R = 1.0

# Radial mesh type: keep radial grid uniform here
rad_unif = 1

# ---------------------------------------------------------------------
# Solution and data functions for Problem 2
#   u(x,y) = (x+1)^{5/2}(y+1)^{5/2} - (x+1)(y+1)^{5/2}
#            - (x+1)^{5/2}(y+1) + (x+1)(y+1)
#
#   f(x,y) = (15/4) * ( - (1+x)*sqrt(1+y)
#                       + (1+x)^{5/2}*sqrt(1+y)
#                       - sqrt(1+x)*(1+y)
#                       + sqrt(1+x)*(1+y)^{5/2} )
# ---------------------------------------------------------------------
u = lambda x, y: (
    (x + 1.0)**(2.5) * (y + 1.0)**(2.5)
    - (x + 1.0)      * (y + 1.0)**(2.5)
    - (x + 1.0)**(2.5) * (y + 1.0)
    + (x + 1.0)      * (y + 1.0)
)

f = lambda x, y: (15.0 / 4.0) * (
    - (1.0 + x)       * np.sqrt(1.0 + y)
    + (1.0 + x)**2.5  * np.sqrt(1.0 + y)
    - np.sqrt(1.0 + x) * (1.0 + y)
    + np.sqrt(1.0 + x) * (1.0 + y)**2.5
)

# --- Dirichlet and Neumann boundary data for Problem 2 ---

# Dirichlet boundary data: u on the boundary
g_dirichlet = lambda x, y: u(x, y)

# Partial derivatives of u
def u_x(x, y):
    a = x + 1.0
    b = y + 1.0
    # u = a^{5/2} b^{5/2} - a b^{5/2} - a^{5/2} b + a b
    term1 = 2.5 * a**1.5 * b**2.5          # d/dx of a^{5/2} b^{5/2}
    term2 = - b**2.5                       # d/dx of - a b^{5/2}
    term3 = - 2.5 * a**1.5 * b             # d/dx of - a^{5/2} b
    term4 = b                              # d/dx of a b
    return term1 + term2 + term3 + term4

def u_y(x, y):
    a = x + 1.0
    b = y + 1.0
    # u = a^{5/2} b^{5/2} - a b^{5/2} - a^{5/2} b + a b
    term1 = 2.5 * a**2.5 * b**1.5          # d/dy of a^{5/2} b^{5/2}
    term2 = - 2.5 * a * b**1.5             # d/dy of - a b^{5/2}
    term3 = - a**2.5                       # d/dy of - a^{5/2} b
    term4 = a                              # d/dy of a b
    return term1 + term2 + term3 + term4

# Neumann boundary data: ∂u/∂n on r = 1 (n = (x, y) on unit circle)
g_neumann = lambda x, y: u_x(x, y) * x + u_y(x, y) * y

# N and M values (Problem 2: Table 3 uses fixed N = 64, varying M)
M_values = [64, 128, 256, 512]
N_fixed  = 64

# ----------------------------------------------------------
# Methods to test
# ----------------------------------------------------------
methods = [
    dict(
        name="uniform_fft",
        label="Uniform Mesh",
        azu_unif=2,
        mesh_kind=None,
        use_nudft=None,
    ),
    dict(
        name="Fixed Nonuniform Mesh - NUDFT",
        label="Fixed Nonuniform Mesh - NUDFT",
        azu_unif=1,
        mesh_kind="jittered",
        use_nudft=True,
    ),
    dict(
        name="Fixed Nonuniform Mesh - NUFFT",
        label="Fixed Nonuniform Mesh - NUFFT",
        azu_unif=1,
        mesh_kind="jittered",
        use_nudft=False,
    ),
]

BC_MAP = {
    "dirichlet": 1,
    "neumann": 2,
}

QUAD_MAP = {
    "trapezoidal": 1,
    "simpson": 2,
}

# Helper Functions

In [ ]:
ANGLE_MESH_CACHE = {}

def get_cached_angle_mesh(method_cfg, N, M):
    azu_unif = method_cfg["azu_unif"]
    mesh_kind = method_cfg["mesh_kind"]
    method_name = method_cfg["name"]

    if azu_unif == 2:
        return generate_uniform_azimuthal(N)

    if azu_unif == 1:
        key = (method_name, N, mesh_kind)
        if key not in ANGLE_MESH_CACHE:
            ANGLE_MESH_CACHE[key] = generate_fixed_nonuniform_azimuthal(
                N, kind=mesh_kind or "rand"
            )
        return ANGLE_MESH_CACHE[key]

    if azu_unif == 0:
        key = (method_name, N, M, mesh_kind)
        if key not in ANGLE_MESH_CACHE:
            ANGLE_MESH_CACHE[key] = generate_nonuniform_azimuthal(
                N, M, kind=mesh_kind or "rand"
            )
        return ANGLE_MESH_CACHE[key]

    raise ValueError("Incorrect index for 'azu_unif'")

def build_radial_mesh(M):
    if rad_unif == 1:
        return generate_uniform_radial(M, R)
    return generate_nonuniform_radial(M, R)

def run_single_case(N, M, method_cfg, bc_name, quad_name):
    bc_choice = BC_MAP[bc_name]
    quad_rule = QUAD_MAP[quad_name]

    azu_unif = method_cfg["azu_unif"]
    use_nudft = method_cfg["use_nudft"]

    iRadius = build_radial_mesh(M)
    iAngle  = get_cached_angle_mesh(method_cfg, N, M)

    x_coord, y_coord = generate_cartesian_grid_on_disk(iAngle, iRadius)

    # interior data and true solution
    f_values = generate_grid_values(f, x_coord, y_coord)
    u_true   = generate_grid_values(u, x_coord, y_coord)

    # --- CHANGED: boundary data depends on BC ---
    if bc_choice == 1:  # Dirichlet
        g_values = generate_grid_values(
            g_dirichlet, x_coord[:, M - 1], y_coord[:, M - 1]
        )
    elif bc_choice == 2:  # Neumann
        g_values = generate_grid_values(
            g_neumann, x_coord[:, M - 1], y_coord[:, M - 1]
        )
    else:
        raise ValueError("Unknown bc_choice")

    # n = 0 mode for Neumann (phi_0), empty for Dirichlet
    if bc_choice == 2:
        u_fourier_0 = u_true.mean(axis=0)
    else:
        u_fourier_0 = np.array([])

    t0 = time.perf_counter()
    try:
        u_approx = poisson_solver(
            f_values, g_values, u_fourier_0,
            N, M, iRadius, iAngle, R,
            quad_rule, bc_choice,
            rad_unif, azu_unif,
            use_nudft_angular=(use_nudft if use_nudft is not None else False),
            maxiter_nufft=50,
            tol_nufft=1e-8,
        )
        runtime = time.perf_counter() - t0

        _, linf_rel, _, l2_rel = compute_error_metrics(
            u_approx, u_true, iRadius, iAngle
        )

    except MemoryError:
        runtime = np.nan
        linf_rel = np.nan
        l2_rel = np.nan

    return {
        "method": method_cfg["name"],
        "label": method_cfg["label"],
        "N": N,
        "M": M,
        "bc": bc_name,
        "quad": quad_name,
        "runtime": runtime,
        "L_inf_rel": linf_rel,
        "L2_rel": l2_rel,
    }

# Run Code, table 3

In [4]:
# Problem 2 – TABLE 3: fixed N = 64, vary M, both BCs, both quadrature rules

table3_results = []

for method in methods:
    print(f"\n{method['label']} -- TABLE 3 (Problem 2)")
    for M in M_values:
        for quad_name in ["trapezoidal", "simpson"]:
            for bc_name in ["dirichlet", "neumann"]:
                res = run_single_case(
                    N=N_fixed,
                    M=M,
                    method_cfg=method,
                    bc_name=bc_name,
                    quad_name=quad_name,
                )
                table3_results.append(res)

                print(
                    f"M={M:4d}, quad={quad_name:12s}, bc={bc_name:9s}, "
                    f"L_inf_rel={res['L_inf_rel']:.3e}, "
                    f"L2_rel={res['L2_rel']:.3e}, "
                    f"time={res['runtime']:.3f} s"
                )

df_table3 = pd.DataFrame(table3_results)


Uniform Mesh -- TABLE 3 (Problem 2)
M=  64, quad=trapezoidal , bc=dirichlet, L_inf_rel=3.246e-05, L2_rel=1.014e-04, time=0.003 s
M=  64, quad=trapezoidal , bc=neumann  , L_inf_rel=2.163e-04, L2_rel=4.759e-04, time=0.004 s
M=  64, quad=simpson     , bc=dirichlet, L_inf_rel=3.089e-06, L2_rel=9.263e-06, time=0.015 s
M=  64, quad=simpson     , bc=neumann  , L_inf_rel=9.832e-07, L2_rel=1.458e-06, time=0.012 s
M= 128, quad=trapezoidal , bc=dirichlet, L_inf_rel=7.989e-06, L2_rel=2.497e-05, time=0.005 s
M= 128, quad=trapezoidal , bc=neumann  , L_inf_rel=5.323e-05, L2_rel=1.172e-04, time=0.003 s
M= 128, quad=simpson     , bc=dirichlet, L_inf_rel=4.076e-07, L2_rel=1.139e-06, time=0.020 s
M= 128, quad=simpson     , bc=neumann  , L_inf_rel=1.530e-07, L2_rel=1.839e-07, time=0.021 s


C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]


M= 256, quad=trapezoidal , bc=dirichlet, L_inf_rel=1.982e-06, L2_rel=6.196e-06, time=0.006 s
M= 256, quad=trapezoidal , bc=neumann  , L_inf_rel=1.320e-05, L2_rel=2.906e-05, time=0.010 s
M= 256, quad=simpson     , bc=dirichlet, L_inf_rel=5.495e-08, L2_rel=1.413e-07, time=0.047 s
M= 256, quad=simpson     , bc=neumann  , L_inf_rel=2.250e-08, L2_rel=2.363e-08, time=0.043 s
M= 512, quad=trapezoidal , bc=dirichlet, L_inf_rel=4.934e-07, L2_rel=1.543e-06, time=0.012 s
M= 512, quad=trapezoidal , bc=neumann  , L_inf_rel=3.288e-06, L2_rel=7.237e-06, time=0.015 s
M= 512, quad=simpson     , bc=dirichlet, L_inf_rel=7.444e-09, L2_rel=1.760e-08, time=0.088 s
M= 512, quad=simpson     , bc=neumann  , L_inf_rel=3.058e-09, L2_rel=3.152e-09, time=0.081 s
M=1024, quad=trapezoidal , bc=dirichlet, L_inf_rel=1.231e-07, L2_rel=3.850e-07, time=0.037 s
M=1024, quad=trapezoidal , bc=neumann  , L_inf_rel=8.200e-07, L2_rel=1.806e-06, time=0.030 s
M=1024, quad=simpson     , bc=dirichlet, L_inf_rel=1.137e-09, L2_rel=2

C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]
C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]
C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]
C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]


M= 256, quad=trapezoidal , bc=dirichlet, L_inf_rel=1.743e-01, L2_rel=2.056e-01, time=0.074 s
M= 256, quad=trapezoidal , bc=neumann  , L_inf_rel=6.134e-01, L2_rel=1.014e+00, time=0.019 s
M= 256, quad=simpson     , bc=dirichlet, L_inf_rel=1.823e-01, L2_rel=2.116e-01, time=0.076 s
M= 256, quad=simpson     , bc=neumann  , L_inf_rel=6.173e-01, L2_rel=1.018e+00, time=0.048 s


...

C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]


M= 512, quad=simpson     , bc=neumann  , L_inf_rel=4.238e-01, L2_rel=5.323e-01, time=5.508 s
M=1024, quad=trapezoidal , bc=dirichlet, L_inf_rel=3.405e-01, L2_rel=3.271e-01, time=8.182 s
M=1024, quad=trapezoidal , bc=neumann  , L_inf_rel=4.240e-01, L2_rel=5.323e-01, time=8.252 s


C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]


M=1024, quad=simpson     , bc=dirichlet, L_inf_rel=3.405e-01, L2_rel=3.271e-01, time=8.477 s
M=1024, quad=simpson     , bc=neumann  , L_inf_rel=4.239e-01, L2_rel=5.323e-01, time=8.376 s


C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]


# View Results

In [5]:
def dash_if_nan(x):
    return "—" if pd.isna(x) else f"{x:.1e}"

for method in methods:
    label = method["label"]
    name  = method["name"]

    print(f"\n{'='*80}")
    print(f"{label} : TABLE 3 (Problem 2)")
    print(f"{'='*80}")

    df3 = df_table3[df_table3["method"] == name].copy()

    cols = []

    # Trapezoidal, Dirichlet
    cols.append(("Trapezoidal rule", "Dirichlet", "||·||∞",
                 df3[(df3["quad"]=="trapezoidal") & (df3["bc"]=="dirichlet")]
                    .set_index("M")["L_inf_rel"]))
    cols.append(("Trapezoidal rule", "Dirichlet", "||·||2",
                 df3[(df3["quad"]=="trapezoidal") & (df3["bc"]=="dirichlet")]
                    .set_index("M")["L2_rel"]))

    # Trapezoidal, Neumann
    cols.append(("Trapezoidal rule", "Neumann", "||·||∞",
                 df3[(df3["quad"]=="trapezoidal") & (df3["bc"]=="neumann")]
                    .set_index("M")["L_inf_rel"]))
    cols.append(("Trapezoidal rule", "Neumann", "||·||2",
                 df3[(df3["quad"]=="trapezoidal") & (df3["bc"]=="neumann")]
                    .set_index("M")["L2_rel"]))

    # Simpson, Dirichlet
    cols.append(("Simpson's rule", "Dirichlet", "||·||∞",
                 df3[(df3["quad"]=="simpson") & (df3["bc"]=="dirichlet")]
                    .set_index("M")["L_inf_rel"]))
    cols.append(("Simpson's rule", "Dirichlet", "||·||2",
                 df3[(df3["quad"]=="simpson") & (df3["bc"]=="dirichlet")]
                    .set_index("M")["L2_rel"]))

    # Simpson, Neumann
    cols.append(("Simpson's rule", "Neumann", "||·||∞",
                 df3[(df3["quad"]=="simpson") & (df3["bc"]=="neumann")]
                    .set_index("M")["L_inf_rel"]))
    cols.append(("Simpson's rule", "Neumann", "||·||2",
                 df3[(df3["quad"]=="simpson") & (df3["bc"]=="neumann")]
                    .set_index("M")["L2_rel"]))

    data   = { (r,bc,norm): series for (r,bc,norm,series) in cols }
    pivot3 = pd.DataFrame(data)
    pivot3.index.name = "M"
    pivot3 = pivot3.reindex(M_values)
    pivot3 = pivot3.sort_index(axis=1, level=[0,1,2])

    display(pivot3.map(dash_if_nan))


Uniform Mesh : TABLE 3 (Problem 2)


Simpson's rule                            Trapezoidal rule           \
          Dirichlet           Neumann                 Dirichlet            
             ||·||2   ||·||∞   ||·||2   ||·||∞           ||·||2   ||·||∞   
M                                                                          
64          9.3e-06  3.1e-06  1.5e-06  9.8e-07          1.0e-04  3.2e-05   
128         1.1e-06  4.1e-07  1.8e-07  1.5e-07          2.5e-05  8.0e-06   
256         1.4e-07  5.5e-08  2.4e-08  2.3e-08          6.2e-06  2.0e-06   
512         1.8e-08  7.4e-09  3.2e-09  3.1e-09          1.5e-06  4.9e-07   
1024        2.2e-09  1.1e-09  8.2e-10  9.9e-10          3.8e-07  1.2e-07   

                        
      Neumann           
       ||·||2   ||·||∞  
M                       
64    4.8e-04  2.2e-04  
128   1.2e-04  5.3e-05  
256   2.9e-05  1.3e-05  
512   7.2e-06  3.3e-06  
1024  1.8e-06  8.2e-07


Fixed Nonuniform Mesh - NUDFT : TABLE 3 (Problem 2)


Simpson's rule                            Trapezoidal rule           \
          Dirichlet           Neumann                 Dirichlet            
             ||·||2   ||·||∞   ||·||2   ||·||∞           ||·||2   ||·||∞   
M                                                                          
64          2.3e-01  1.9e-01  1.1e+00  6.4e-01          2.2e-01  1.6e-01   
128         2.1e-01  1.6e-01  1.1e+00  6.6e-01          2.0e-01  1.6e-01   
256         2.1e-01  1.8e-01  1.0e+00  6.2e-01          2.1e-01  1.7e-01   
512         2.0e-01  1.7e-01  1.0e+00  6.1e-01          2.0e-01  1.7e-01   
1024        2.1e-01  1.7e-01  1.0e+00  6.3e-01          2.1e-01  1.7e-01   

                        
      Neumann           
       ||·||2   ||·||∞  
M                       
64    1.1e+00  6.2e-01  
128   1.1e+00  6.6e-01  
256   1.0e+00  6.1e-01  
512   1.0e+00  6.1e-01  
1024  1.0e+00  6.3e-01


Fixed Nonuniform Mesh - NUFFT : TABLE 3 (Problem 2)


Simpson's rule                            Trapezoidal rule           \
          Dirichlet           Neumann                 Dirichlet            
             ||·||2   ||·||∞   ||·||2   ||·||∞           ||·||2   ||·||∞   
M                                                                          
64          3.3e-01  3.4e-01  5.3e-01  4.2e-01          3.3e-01  3.4e-01   
128         3.3e-01  3.4e-01  5.3e-01  4.2e-01          3.3e-01  3.4e-01   
256         3.3e-01  3.4e-01  5.3e-01  4.2e-01          3.3e-01  3.4e-01   
512         3.3e-01  3.4e-01  5.3e-01  4.2e-01          3.3e-01  3.4e-01   
1024        3.3e-01  3.4e-01  5.3e-01  4.2e-01          3.3e-01  3.4e-01   

                        
      Neumann           
       ||·||2   ||·||∞  
M                       
64    5.3e-01  4.2e-01  
128   5.3e-01  4.2e-01  
256   5.3e-01  4.2e-01  
512   5.3e-01  4.2e-01  
1024  5.3e-01  4.2e-01